# 外部 baseline 真实 GPU 链路测试入口

该 Notebook 用于 Colab 冷启动, 挂载 Google Drive、拉取仓库、按登记表补齐 T2SMark 官方源码缓存、加载 SD3.5 Medium、运行小样本真实 GPU 链路测试, 并把 Tree-Ring、Gaussian Shading、Shallow Diffuse 和 T2SMark 四个主表 external baseline 接入统一 adapter 命令计划。全部核对文件会打包保存到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/external_baseline_gpu_smoke.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 `stabilityai/stable-diffusion-3.5-medium` 访问权限, 并配置 `HF_TOKEN`。
3. 默认会在 `/content/drive/MyDrive/SLM/external_baseline_gpu_smoke` 查找历史结果包; 只有当历史 T2SMark `results.json` 至少包含 5 个数字样本条目时才会复用。
4. 若历史结果包不存在或样本数不足, 本入口会按 `external_baseline/source_registry.json` 下载 T2SMark 官方源码并生成新的 5 样本真实 GPU 结果。
5. Tree-Ring 默认执行 SD3.5 method-faithful adapter; Gaussian Shading 与 Shallow Diffuse 当前仍执行 SD3.5 latent 级 GPU adapter。该入口验证模型主线形状、GPU 张量路径和 observation 落盘协议, 不声明论文级外部 baseline 对比结论。
6. 默认共享样本数为 5, 与当前 T2SMark 链路测试一致。
7. 本入口会先打包所有核对文件, 再执行断言, 便于失败时回传诊断结果。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_EXTERNAL_BASELINE_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/external_baseline_gpu_smoke')
os.environ.setdefault('SLM_WM_EXTERNAL_BASELINE_PRIOR_DRIVE_DIR', '/content/drive/MyDrive/SLM/external_baseline_gpu_smoke')
os.environ.setdefault('SLM_WM_T2SMARK_MODEL_ID', 'stabilityai/stable-diffusion-3.5-medium')
os.environ.setdefault('SLM_WM_T2SMARK_RUN_NAME', 't2smark_sd35_medium_gpu_smoke')
os.environ.setdefault('SLM_WM_T2SMARK_ROBUST_TEST_NUM', '5')
os.environ.setdefault('SLM_WM_T2SMARK_CLIP_TEST_NUM', '0')
os.environ.setdefault('SLM_WM_T2SMARK_NUM_INFERENCE_STEPS', '8')
os.environ.setdefault('SLM_WM_T2SMARK_NUM_INVERSION_STEPS', '3')
os.environ.setdefault('SLM_WM_T2SMARK_GUIDANCE_SCALE', '4.0')
os.environ.setdefault('SLM_WM_EXTERNAL_BASELINE_REUSE_EXISTING', '1')
os.environ.setdefault('SLM_WM_EXTERNAL_BASELINE_REUSE_DRIVE', '1')
os.environ.setdefault('SLM_WM_EXTERNAL_BASELINE_REQUIRE_CUDA', '1')
os.environ.setdefault('SLM_WM_PRIMARY_BASELINE_MAX_SAMPLES', '5')
os.environ.setdefault('SLM_WM_TREE_RING_ADAPTER_MODE', 'method_faithful_sd35')
os.environ.setdefault('SLM_WM_TREE_RING_ATTACK_FAMILIES', '')
print({
    'drive_output_dir': os.environ['SLM_WM_EXTERNAL_BASELINE_DRIVE_OUTPUT_DIR'],
    'prior_drive_dir': os.environ['SLM_WM_EXTERNAL_BASELINE_PRIOR_DRIVE_DIR'],
    'model_id': os.environ['SLM_WM_T2SMARK_MODEL_ID'],
    'robust_test_num': os.environ['SLM_WM_T2SMARK_ROBUST_TEST_NUM'],
    'num_inference_steps': os.environ['SLM_WM_T2SMARK_NUM_INFERENCE_STEPS'],
    'num_inversion_steps': os.environ['SLM_WM_T2SMARK_NUM_INVERSION_STEPS'],
    'primary_baseline_max_samples': os.environ['SLM_WM_PRIMARY_BASELINE_MAX_SAMPLES'],
    'tree_ring_adapter_mode': os.environ['SLM_WM_TREE_RING_ADAPTER_MODE'],
    'tree_ring_attack_families': os.environ['SLM_WM_TREE_RING_ATTACK_FAMILIES'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub open_clip_torch scikit-learn scipy pandas datasets tqdm


In [ ]:
import importlib.metadata as importlib_metadata
import importlib.util

import diffusers
import transformers


def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'


dependency_report = {
    'diffusers': diffusers.__version__,
    'transformers': transformers.__version__,
    'accelerate': package_version_or_missing('accelerate'),
    'huggingface_hub': package_version_or_missing('huggingface_hub'),
    'safetensors': package_version_or_missing('safetensors'),
    'sentencepiece': package_version_or_missing('sentencepiece'),
    'protobuf': package_version_or_missing('protobuf'),
    'open_clip_torch': package_version_or_missing('open_clip_torch'),
    'scikit_learn': package_version_or_missing('scikit-learn'),
    'scipy': package_version_or_missing('scipy'),
    'pandas': package_version_or_missing('pandas'),
    'datasets': package_version_or_missing('datasets'),
    'tqdm': package_version_or_missing('tqdm'),
    'torchvision_available': importlib.util.find_spec('torchvision') is not None,
    'pillow_available': importlib.util.find_spec('PIL') is not None,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from paper_workflow.colab_utils.external_baseline_gpu_smoke import run_default_external_baseline_gpu_smoke_plan

external_baseline_gpu_smoke_summary = run_default_external_baseline_gpu_smoke_plan(root='.')
external_baseline_gpu_smoke_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/external_baseline_gpu_smoke/external_baseline_gpu_smoke_summary.json')
prior_manifest_path = Path('outputs/external_baseline_gpu_smoke/external_baseline_gpu_smoke_prior_package_manifest.json')
source_prepare_path = Path('outputs/external_baseline_gpu_smoke/t2smark_source_prepare_result.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else external_baseline_gpu_smoke_summary)
for diagnostic_path in (prior_manifest_path, source_prepare_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8'))


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.external_baseline_gpu_smoke import package_external_baseline_gpu_smoke_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ.get('SLM_WM_EXTERNAL_BASELINE_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/external_baseline_gpu_smoke')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'external_baseline_gpu_smoke_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_external_baseline_gpu_smoke_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/external_baseline_gpu_smoke').glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

for result_path in sorted(Path('outputs/external_baseline_gpu_smoke').glob('*.csv')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

observations_path_text = external_baseline_gpu_smoke_summary.get('baseline_observations_path') or ''
if observations_path_text:
    observations_path = Path(observations_path_text)
    if observations_path.is_file():
        print(observations_path.read_text(encoding='utf-8')[:4000])

assert external_baseline_gpu_smoke_summary['run_decision'] == 'pass', external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['external_baseline_gpu_smoke_ready'] is True, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['t2smark_real_gpu_smoke_ready'] is True, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['adapter_execution_ready'] is True, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['adapter_observation_count'] > 0, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['primary_baseline_adapter_ready'] is True, external_baseline_gpu_smoke_summary
assert set(external_baseline_gpu_smoke_summary['primary_baseline_ids']) == {'tree_ring', 'gaussian_shading', 'shallow_diffuse', 't2smark'}, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['expected_sample_count'] == 5, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['t2smark_result_count'] >= 5, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['image_pair_count'] >= 5, external_baseline_gpu_smoke_summary
assert external_baseline_gpu_smoke_summary['primary_baseline_observation_count'] >= 40, external_baseline_gpu_smoke_summary
